# Time Windows Analysis

**Purpose:** Inspect time-windowed correlation networks and clustering behavior.
**Inputs:** SEEG `.mat` files in `data/stereoeeg_patients/`.
**Outputs:** Figures and Sankey HTML in `data/figures/time_windows_analysis/`.
**Date:** 2025-12-11


In [ ]:
# %% Configuration
CONFIG = {
    "patient": "Pat_02",
    "phase": "rsPre",
    "band": "beta",
    "n_intervals": 30,
    "interval_index": 0,
    "max_plots": 12,
    "output_dir": "figures/time_windows_analysis",
    "correlation_protocol": {"filter_type": "abs", "spectral_cleaning": True, "threshold": 0},
    "filter_order": 1,
    "linkage_method": "ward",
    "scaling_factor": 1.0,
}


In [ ]:
# %% Setup
from lrgsglib import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from scipy.cluster.hierarchy import dendrogram

from lrg_eegfc.notebook import *
from lrg_eegfc.utils.corrmat.network import process_network_for_phase
from lrgsglib.utils.lrg import compute_laplacian_properties

path_figs = setup_notebook(CONFIG["output_dir"])


## 1. Load Data and Split into Time Windows

Split a single phase into fixed-length intervals and build correlation networks.

In [ ]:
data_dict, int_label_map = load_data_dict(
    pat_list=[CONFIG["patient"]],
    phase_labels=[CONFIG["phase"]],
)
entry = data_dict[CONFIG["patient"]][CONFIG["phase"]]
data_ts = entry["data"]
fs_raw = entry.get("fs", None)
fs = float(np.asarray(fs_raw).flat[0]) if fs_raw is not None else 1024.0
pin_labels = int_label_map[CONFIG["patient"]]["label"]

interval_length = data_ts.shape[1] // CONFIG["n_intervals"]
interval_sec = interval_length / fs
print(f"Interval length: {interval_length} samples ({interval_sec:.2f} s)")

interval_results = []
for idx in range(CONFIG["n_intervals"]):
    start = idx * interval_length
    end = (idx + 1) * interval_length
    segment = data_ts[:, start:end]

    G, label_dict, lnkgM, clTh, corr_mat, dists = process_network_for_phase(
        segment,
        fs,
        CONFIG["band"],
        CONFIG["correlation_protocol"],
        pin_labels,
        filter_order=CONFIG["filter_order"],
        linkage_method=CONFIG["linkage_method"],
        scaling_factor=CONFIG["scaling_factor"],
    )

    if G is None or lnkgM is None:
        continue

    interval_results.append(
        {
            "interval": idx,
            "G": G,
            "label_dict": label_dict,
            "linkage_matrix": lnkgM,
            "cl_threshold": clTh,
            "corr_mat": corr_mat,
            "dists": dists,
        }
    )

print(f"✓ Built {len(interval_results)} interval networks")


## 2. Dendrograms Across Intervals

Visualize clustering structure for the first N intervals.

In [ ]:
def plot_interval_dendrograms(results, max_plots=12):
    if not results:
        print("No interval results available.")
        return

    n_plots = min(len(results), max_plots)
    ncols = 4
    nrows = int(np.ceil(n_plots / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for ax in axes[n_plots:]:
        ax.axis("off")

    for idx, result in enumerate(results[:n_plots]):
        lnkgM = result["linkage_matrix"]
        clTh = result["cl_threshold"]
        dendrogram(
            lnkgM,
            ax=axes[idx],
            color_threshold=clTh,
            above_threshold_color="k",
            orientation="right",
            no_labels=True,
        )
        axes[idx].axvline(clTh, color="b", linestyle="--", linewidth=1)
        axes[idx].set_xscale("log")
        axes[idx].set_title(f"Interval {result['interval'] + 1}", fontsize=10)

    outfile = path_figs / "interval_dendrograms.png"
    fig.tight_layout()
    fig.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {outfile}")

plot_interval_dendrograms(interval_results, max_plots=CONFIG["max_plots"])


## 3. Outlier-Aware Clustering for a Single Interval

Compare dendrogram-consistent clusters and outlier-aware clustering.

In [ ]:
if not interval_results:
    print("No interval results available.")
else:
    idx = min(CONFIG["interval_index"], len(interval_results) - 1)
    result = interval_results[idx]
    lnkgM = result["linkage_matrix"]
    clTh = result["cl_threshold"]

    fig, ax = plt.subplots(figsize=(8, 5))
    dendro = dendrogram(
        lnkgM,
        ax=ax,
        color_threshold=clTh,
        above_threshold_color="k",
        no_labels=True,
    )
    ax.axhline(clTh, color="r", linestyle="--", linewidth=1)
    ax.set_yscale("log")
    ax.set_title(f"Interval {result['interval'] + 1} Dendrogram")

    clusters_dendro = get_dendrogram_consistent_clusters(lnkgM, dendro, clTh)
    clusters_outliers = fcluster_with_outliers(lnkgM, clTh, outlier_sensitivity=1.5, min_cluster_size=2)

    outliers_dendro = np.where(clusters_dendro == 0)[0]
    outliers_auto = np.where(clusters_outliers == 0)[0]

    print(f"Dendrogram outliers: {outliers_dendro}")
    print(f"Auto outliers: {outliers_auto}")

    # Optional: visualize clusters on the network
    G = result["G"]
    node_order = list(G.nodes())
    node_labels = {n: result["label_dict"].get(n, str(n)) for n in node_order}

    dendro_map = {node_order[i]: clusters_dendro[i] for i in range(len(node_order))}
    colors = [dendro_map[n] for n in node_order]

    fig2, ax2 = plt.subplots(figsize=(6, 5))
    nx.draw(
        G,
        ax=ax2,
        node_color=colors,
        node_size=40,
        with_labels=False,
    )
    ax2.set_title("Dendrogram-Consistent Clusters")
    fig2.tight_layout()
    outfile = path_figs / f"interval_{result['interval'] + 1}_clusters.png"
    fig2.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {outfile}")


## 4. Metastable Clustering Across Tau

Generate a Sankey diagram showing cluster evolution across diffusion times.

In [ ]:
if not interval_results:
    print("No interval results available.")
else:
    result = interval_results[min(CONFIG["interval_index"], len(interval_results) - 1)]
    G = result["G"]

    spectrum, _, _, _, _ = compute_laplacian_properties(G, tau=None)
    lmax = float(np.max(spectrum)) if len(spectrum) else 1.0
    tau_values = np.array([0.0, 1.0 / lmax, 2.0 / lmax, 5.0 / lmax, 10.0 / lmax])

    partitions, n_clusters = compute_clustering_across_tau(
        result["linkage_matrix"],
        tau_values,
        G,
        method=CONFIG["linkage_method"],
        scaling_factor=CONFIG["scaling_factor"],
    )

    node_order = list(G.nodes())
    node_labels = [result["label_dict"].get(n, str(n)) for n in node_order]

    fig = create_sankey_diagram(
        partitions,
        tau_values,
        node_labels=node_labels,
        title=f"{CONFIG['patient']} {CONFIG['phase']} {CONFIG['band']} — Cluster Evolution",
    )

    outfile = path_figs / f"{CONFIG['patient']}_{CONFIG['phase']}_{CONFIG['band']}_sankey.html"
    fig.write_html(outfile)
    fig
    print(f"Saved: {outfile}")
